In [2]:
!pip install langchain langchain-community langchain-huggingface
!pip install faiss-cpu sentence-transformers pypdf transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.16
    Uninstalling langchain-core-1.2.16:
      Successfully uninstalled langchain-core-1.2.16
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is 

In [3]:
from google.colab import files
uploaded = files.upload()

Saving Annual-Report-FY-2023-24.pdf to Annual-Report-FY-2023-24.pdf


In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("Annual-Report-FY-2023-24.pdf")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)

docs = splitter.split_documents(documents)

print("Total chunks:", len(docs))

Total chunks: 91


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k":3})

In [7]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=120,
    do_sample=True,
    temperature=0.2,
    pad_token_id=tokenizer.eos_token_id
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [12]:
def ask_question(query):

    docs = retriever.invoke(query)

    context = "\n\n".join([doc.page_content for doc in docs])

    pages = [doc.metadata.get("page", "Unknown") for doc in docs]

    prompt = f"""
You are a question answering system.

Use the provided context to answer the question.

Respond with ONLY the final answer in one short sentence.
Do not generate additional questions.

Context:
{context}

Question: {query}

Answer:
"""

    result = generator(prompt)

    output = result[0]["generated_text"]

    # remove prompt
    answer = output.replace(prompt, "").strip()

    # stop generation after first line
    answer = answer.split("\n")[0]

    print("\n==============================")
    print("Question:", query)
    print("==============================\n")

    print("Answer:")
    print(answer)

    print("\nSources (Page Numbers):")
    print(", ".join([f"Page {p}" for p in pages]))

    print("\nRetrieved Context:\n")

    for i, doc in enumerate(docs):
        print(f"Page {pages[i]}")
        print(doc.page_content[:400])
        print("\n------------------------\n")

In [13]:
ask_question("Who is the CEO of Swiggy?")

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question: Who is the CEO of Swiggy?

Answer:
Sriharsha Majety is the CEO of Swiggy.

Sources (Page Numbers):
Page 30, Page 27, Page 2

Retrieved Context:

Page 30
2. Names of associates or joint ventures which have been liquidated or sold during the year: NIL  
For and on Behalf of the Board of Directors of  
Swiggy Limited 
 
 
 
          Sd/-                                          Sd/-                                               Sd/-                                Sd/- 
 
Sriharsha Majety  
Managing Director & 
Group CEO 
(DIN: 06680073) 
 
Lakshmi N

------------------------

Page 27
review. 
 
21. ACKNOWLEDGEMENTS 
 
Your Directors place on record their sincere thanks to bankers, business associates, consultants, and various 
Government Authorities for their continued support extended to your Companies activities during the year 
under review. Your Directors also acknowledges gratefully the shareholders for their support and  
confidence reposed on your Company.  
 
Behalf of

In [14]:
ask_question("What services does Swiggy provide?")

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question: What services does Swiggy provide?

Answer:
Swiggy offers Food Delivery, Quick Commerce, and Instamart services.

Sources (Page Numbers):
Page 5, Page 5, Page 7

Retrieved Context:

Page 5
cost of the Company. Your directors are confident that the Company will grow and prosper in the coming 
years. 
 
2. RESULTS OF OPERATIONS AND THE STATE OF COMPANY’S AFFAIRS  
 
History: 
 
At Swiggy, our mission is to elevate the quality of life for urban consumers by offering unparalleled 
convenience. Innovation has been an integral part of our DNA which encourages us to ideate, experiment 
and

------------------------

Page 5
India, launching Food Delivery in 2014. Today, Swiggy offers its Food Delivery service in 653 cities across 
India serving ~13mn users 1 through a wide network of 196k restaurant partners 2. Our sharp focus on 
innovation coupled with strong execution yielded yet another milestone - we became one of the very few 
global food delivery platforms to achieve EBITDA p